In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [ ]:
load_dotenv(override=True)

anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")

In [ ]:
openai = OpenAI()
anthropic = OpenAI(api_key=anthropic_api_key, base_url="https://api.anthropic.com/v1/")
ollama = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

GPT_MODEL = "gpt-4.1-mini"
CLAUDE_MODEL = "claude-haiku-4-5"
OLLAMA_MODEL = "llama3.2"

In [ ]:
# Three local fishermen down at a UK harbour. Alex and Charlie argue with each other;
# Blake tries to keep the peace between them.

alex_system_prompt = """
You are Alex, a fisherman from a small harbour town on the UK coast. You've been hauling pots and nets for thirty years and reckon you know best about everything - the weather, the tides, the going rate for the day's catch. You're argumentative and snarky, always ready to disagree with whatever the other two say, in a blunt, down-to-earth way, with the odd bit of local fishing chat thrown in. Keep your replies short and natural, like real quayside talk.
You are in a conversation with Blake and Charlie, fellow fishermen down at the harbour.
"""

charlie_system_prompt = """
You are Charlie, a stubborn old fisherman from the same UK harbour. You've fished these waters your whole life and don't take kindly to being told you're wrong. You argue back at whatever Alex says and refuse to back down, in a blunt, combative way, with talk of nets, tides, and the day's catch. Keep your replies short and natural, like real quayside talk.
You are in a conversation with Alex and Blake, fellow fishermen down at the harbour.
"""

blake_system_prompt = """
You are Blake, a good-natured fisherman from the same UK harbour as Alex and Charlie. You're the calm one of the three - always trying to smooth things over, find common ground, and keep the peace when Alex and Charlie start bickering, usually over a cup of tea on the quayside. Keep your replies short and natural, like real quayside talk.
You are in a conversation with Alex and Charlie, fellow fishermen down at the harbour.
"""

In [ ]:
PARTICIPANTS = {
    "Alex": {
        "client": openai,
        "model": GPT_MODEL,
        "system_prompt": alex_system_prompt,
        "others": "Blake and Charlie",
    },
    "Charlie": {
        "client": ollama,
        "model": OLLAMA_MODEL,
        "system_prompt": charlie_system_prompt,
        "others": "Alex and Blake",
    },
    "Blake": {
        "client": anthropic,
        "model": CLAUDE_MODEL,
        "system_prompt": blake_system_prompt,
        "others": "Alex and Charlie",
    },
}

history = []  # list of (name, message) tuples

In [ ]:
def render_conversation() -> str:
    """Render the conversation so far as 'Name: message' lines."""
    return "\n".join(f"{name}: {message}" for name, message in history)


def next_message(name: str) -> str:
    """Ask the participant `name` for their next line, given the conversation so far."""
    participant = PARTICIPANTS[name]
    conversation = render_conversation()
    user_prompt = f"""
You are {name}, in conversation with {participant['others']}.
The conversation so far is as follows:
{conversation}
Now with this, respond with what you would like to say next, as {name}.
"""
    messages = [
        {"role": "system", "content": participant["system_prompt"]},
        {"role": "user", "content": user_prompt},
    ]
    response = participant["client"].chat.completions.create(
        model=participant["model"], messages=messages
    )
    return response.choices[0].message.content


def add_message(name: str, message: str) -> None:
    """Append a message to the conversation and display it."""
    history.append((name, message))
    display(Markdown(f"### {name}:\n{message}\n"))

In [ ]:
add_message("Alex", "Mornin'. Nets were empty again out past the point.")
add_message("Charlie", "Empty? You were fishing the wrong ground again, weren't you.")
add_message("Blake", "Now now, let's not start already, it's barely gone six.")

In [ ]:
for round_number in range(3):
    for name in PARTICIPANTS:
        add_message(name, next_message(name))